# 03 · 组合优化与风险分析

**学习目标**：
1. 用均值-方差框架求最小波动组合和最大夏普组合；
2. 看懂**风险贡献**：权重相同不代表风险相同；
3. 用历史模拟法算 VaR 和 CVaR。

**数据**：`scripts/optimize_portfolio.py` 内置的可复现示例价格，无需联网。装了 `pyportfolioopt` 会用它求解，否则用 numpy 自己算。

> 本仓库仅供学习，所有内容不构成投资建议。

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 让 Notebook 能直接复用 scripts/ 里的函数
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
def perf_stats(daily_ret: pd.Series, periods: int = 252) -> pd.Series:
    """年化收益、年化波动、夏普（无风险利率按 0）、最大回撤。"""
    daily_ret = daily_ret.dropna()
    nav = (1 + daily_ret).cumprod()
    ann_ret = nav.iloc[-1] ** (periods / len(daily_ret)) - 1
    ann_vol = daily_ret.std() * np.sqrt(periods)
    max_dd = (nav / nav.cummax() - 1).min()
    return pd.Series({"ann_return": ann_ret, "ann_vol": ann_vol,
                      "sharpe": ann_ret / ann_vol if ann_vol else np.nan, "max_drawdown": max_dd})

In [ ]:
from optimize_portfolio import load_prices

prices = load_prices()
rets = prices.pct_change().dropna()
mu = rets.mean() * 252                 # 年化期望收益（历史均值）
cov = rets.cov() * 252                 # 年化协方差
mu, cov

## 1. 最小波动与最大夏普

最小波动组合有解析解 `w ∝ Σ⁻¹·1`；最大夏普（无风险利率按 0、允许做空时）`w ∝ Σ⁻¹·μ`。下面用 numpy 直接算，并在装了 PyPortfolioOpt 时与其不允许做空的结果对照。

In [ ]:
inv = np.linalg.inv(cov.values)
ones = np.ones(len(mu))
w_minvol = pd.Series(inv @ ones / (ones @ inv @ ones), index=mu.index)
w_tangent = pd.Series(inv @ mu.values / (ones @ inv @ mu.values), index=mu.index)

weights = pd.DataFrame({"min_vol (numpy)": w_minvol, "max_sharpe (numpy, shorting allowed)": w_tangent,
                        "equal_weight": 1 / len(mu)})
try:
    from pypfopt import EfficientFrontier
    ef = EfficientFrontier(mu, cov)
    ef.min_volatility()
    weights["min_vol (PyPortfolioOpt, long-only)"] = pd.Series(ef.clean_weights())
except ImportError:
    print("未安装 pyportfolioopt，只展示 numpy 结果")
weights

In [ ]:
def port_stats(w):
    r = rets @ w
    return perf_stats(r)

pd.DataFrame({name: port_stats(weights[name].values) for name in weights.columns})

**注意看“允许做空的最大夏普”那一列**：它用几倍杠杆做多 A、做空 B，样本内年化收益高得离谱。原因是它把 250 天的历史均值当成了未来收益的准确预测，而 B 恰好在这段时间大跌。真实中这种权重几乎必然在样本外失效，这就是均值-方差优化“对输入极度敏感”的问题。实务中通常禁止做空、限制单一权重，或改用不依赖收益预测的最小波动、风险平价。

## 2. 风险贡献

资产 i 对组合波动的贡献 `RC_i = w_i · (Σw)_i / σ_p`，所有 RC 加起来等于组合波动。等权组合里，波动更大的资产承担了超过 1/N 的风险；而在最小波动组合中，每个资产的风险贡献比例恰好等于它的权重（最小方差组合的性质，可以对照上面的权重表验证）。

In [ ]:
def risk_contrib(w):
    w = np.asarray(w)
    sigma_p = np.sqrt(w @ cov.values @ w)
    rc = w * (cov.values @ w) / sigma_p
    return pd.Series(rc / sigma_p, index=mu.index)    # 占组合风险的比例

pd.DataFrame({"equal_weight": risk_contrib(weights["equal_weight"]),
              "min_vol": risk_contrib(weights["min_vol (numpy)"])})

## 3. 历史模拟 VaR / CVaR

- 95% 日 VaR：历史上最差的 5% 交易日里，亏损的门槛值
- 95% 日 CVaR：最差 5% 交易日的平均亏损，比 VaR 更能反映尾部风险

In [ ]:
port = rets @ weights["equal_weight"].values
var95 = -np.percentile(port, 5)
cvar95 = -port[port <= -var95].mean()
print(f"95% 日 VaR : {var95:.2%}")
print(f"95% 日 CVaR: {cvar95:.2%}")

ax = port.hist(bins=40, figsize=(8, 3))
ax.axvline(-var95, color="red", linestyle="--", label="VaR 95%")
ax.set_title("Daily return distribution (equal weight)")
ax.legend()
plt.show()

## 练习

1. 只用前 125 天估计 μ 和 Σ，再看后 125 天的表现：允许做空的最大夏普组合在样本外还成立吗？
2. 实现风险平价：迭代调整权重，使每个资产的风险贡献相等。
3. 用 `scripts/optimize_portfolio.py` 跑一遍，对照 `reports/portfolio_weights.csv` 和这里的结果。